# PA 766 | 2. Turn text into CSV files for annotation

**Goal:** divide the text files from Notebook 1 into manageable chunks and export one annotation CSV per source file.

Open this notebook in **Google Colab** and run the cells in order. Connect Google Drive and choose the folder containing your `.txt` files. Inspect the chunks, save the CSV files, and then use the annotation guide at the bottom. This notebook needs no API key.

We will practice one dimension from the CARESL annotation guide: **Statement status**. The question is: **How does this passage frame a concrete action for managing electricity demand or load growth?** LLM annotation is a later step and is not part of this notebook.


## 1. Read your text files from Google Drive


In [ ]:
from pathlib import Path  # Work with file and folder paths.
import pandas as pd  # Work with tables of data using the short name pd.
from IPython.display import display  # Show formatted tables in the notebook.
import re  # Find likely sentence boundaries using punctuation.


In [ ]:
from google.colab import drive  # Access Google Drive from Colab.
from pathlib import Path  # Work with folder and file paths.

drive.mount("/content/drive")  # Connect your Google Drive; follow the authorization prompt.

In [9]:
text_folder = Path("/content/drive/MyDrive/a-ncsu-courses/Temporary")
csv_folder = Path("/content/drive/MyDrive/a-ncsu-courses/Temporary/CSV") # Specify the folder containing the converted text files.
texts = {}  # Store each document's text, using its filename as the key.

if not text_folder.is_dir():  # Check that the folder exists.
    raise FileNotFoundError(f"Folder not found: {text_folder}")  # Stop if the folder path is incorrect.

for path in sorted(text_folder.iterdir()):  # Examine files directly inside the folder in filename order.
    if path.is_file() and path.suffix.lower() == ".txt":  # Keep only text files, ignoring capitalization.
        texts[path.name] = path.read_text(encoding="utf-8-sig")  # Read the text, preserving page separators and removing any UTF-8 byte-order mark.

if not texts:  # Check whether any text files were loaded.
    raise ValueError(f"No .txt files found in: {text_folder}")  # Stop if the folder contains no text files.

print(f"Loaded {len(texts)} text file(s):")  # Display the number of loaded documents.
for name in sorted(texts):  # Loop through the filenames in alphabetical order.
    print(name)  # Display each filename.

Loaded 4 text file(s):
D00000-ElectricVehicleScenarioAnalysisWorkshopSeries.txt
D00001-TheEraOfFlatPowerDemandIsOver.txt
D00002-CharacteristicsAndRiskOfEmergingLargeLoads.txt
D00003-PlanningForAndManagingInterminateElecricLoads.txt


## 2. Choose a chunk size

We combine consecutive sentences into chunks of up to 180 words within each PDF page. A sentence longer than 180 words stays intact in its own chunk. Chunks do not overlap, and all extracted words remain in their original order.

The word limit is a target reading length, not a token limit or a guarantee of a complete idea. Sentence boundaries are detected using punctuation, so abbreviations can cause incorrect splits. Sentences that continue onto another page remain separated at the page boundary. Headings, references, and table fragments may also appear in chunks. Inspect the text before assigning a label.

Start with 180 words. You can try another size before annotation begins. Changing the input text or chunk size changes the rows and their IDs; keep the original CSV once you begin labeling.

In [ ]:
MAX_WORDS = 180  # Set the target maximum number of words per chunk.
if not isinstance(MAX_WORDS, int) or isinstance(MAX_WORDS, bool) or MAX_WORDS < 1:  # Require a positive whole number.
    raise ValueError("MAX_WORDS must be a positive whole number.")  # Stop if the setting is invalid.

def chunk_page(page_text, max_words):  # Split one page into chunks without cutting detected sentences.
    chunks = []  # Store completed chunks.
    current = []  # Store sentences for the current chunk.
    current_words = 0  # Track the current chunk's word count.
    text = " ".join(page_text.split())  # Replace line breaks and repeated whitespace with single spaces.
    sentences = re.split(r"(?<=[.!?])\s+", text)  # Split after sentence-ending punctuation followed by whitespace.

    for sentence in sentences:  # Process each detected sentence.
        sentence = sentence.strip()  # Remove surrounding whitespace.
        if not sentence:  # Skip empty sentences.
            continue
        word_count = len(sentence.split())  # Count the words in this sentence.

        if current and current_words + word_count > max_words:  # Start a new chunk if adding this sentence exceeds the target.
            chunks.append(" ".join(current))  # Save the current chunk.
            current = []  # Start an empty chunk.
            current_words = 0  # Reset its word count.

        current.append(sentence)  # Add the entire sentence, even if it alone exceeds the target.
        current_words += word_count  # Update the current chunk's word count.

    if current:  # Check for a final unfinished chunk.
        chunks.append(" ".join(current))  # Save the remaining sentences.

    return chunks  # Return the completed chunks.

## 3. Create the dataset

Each row is one chunk. We retain the source filename and the PDF page number so you can return to the original. `statement_status` and `notes` start blank; you will fill them manually in a spreadsheet.

Page numbers depend on the `\f` page separators written by Notebook 1. Keep those files unchanged. Ordinary text files without these separators will be treated as one page.

In [10]:
rows = []  # Collect one record per chunk.

for filename, text in sorted(texts.items()):  # Process each source file.
    for page_number, page_text in enumerate(text.split("\f"), start=1):  # Split and number the pages.
        chunks = chunk_page(page_text, MAX_WORDS)  # Group detected sentences into chunks.
        for chunk_number, chunk in enumerate(chunks, start=1):  # Number the chunks on each page.
            rows.append({
                "chunk_id": f"{Path(filename).stem}_p{page_number:03d}_c{chunk_number:02d}",  # Identify the source, page, and chunk.
                "source_file": filename,  # Keep the source filename.
                "page": page_number,  # Keep the PDF page number.
                "text": chunk,  # Store the passage.
                "word_count": len(chunk.split()),  # Count its words.
                "statement_status": "",  # Leave the annotation label blank.
                "notes": "",  # Leave space for notes.
            })

if not rows:  # Stop if there is no text to annotate.
    raise ValueError("The text files contain no usable text.")

dataset = pd.DataFrame(rows)  # Create the annotation table.
print(f"Created {len(dataset)} chunks from {len(texts)} document(s).")  # Summarize the result.
display(dataset.head(5))  # Preview the first five chunks.

Created 271 chunks from 4 document(s).


,chunk_id,source_file,page,text,word_count,statement_status,notes
0,D00000-ElectricVehicleScenarioAnalysisWorkshop...,D00000-ElectricVehicleScenarioAnalysisWorkshop...,1,Electric Vehicle Scenario Analysis Workshop Se...,87,,
1,D00000-ElectricVehicleScenarioAnalysisWorkshop...,D00000-ElectricVehicleScenarioAnalysisWorkshop...,2,S T R AT E G I C P L A N N I N G A N D FA C I ...,151,,
2,D00000-ElectricVehicleScenarioAnalysisWorkshop...,D00000-ElectricVehicleScenarioAnalysisWorkshop...,2,The collective sharing of knowledge and experi...,177,,
3,D00000-ElectricVehicleScenarioAnalysisWorkshop...,D00000-ElectricVehicleScenarioAnalysisWorkshop...,2,• Cooperatives are starting to consider resid...,180,,
4,D00000-ElectricVehicleScenarioAnalysisWorkshop...,D00000-ElectricVehicleScenarioAnalysisWorkshop...,3,S T R AT E G I C P L A N N I N G A N D FA C I ...,157,,


## 4. Inspect a chunk

Change `ROW_NUMBER` to inspect a different row. The first row is numbered 0 in Python. Use `source_file` and `page` to find it in the PDF. A cover or contents page is not necessarily a useful practice passage.

In [11]:
ROW_NUMBER = 0  # Choose a row by position; 0 is the first row, 1 is the second, and so on.
row = dataset.iloc[ROW_NUMBER]  # Retrieve the selected row from the dataset.
print(row["chunk_id"])  # Display the chunk's unique ID.
print(f"Source: {row['source_file']} | PDF page: {row['page']} | Words: {row['word_count']}\n")  # Display the source file, page number, and word count, followed by a blank line.
print(row["text"])  # Display the full chunk text for reading and annotation.

D00000-ElectricVehicleScenarioAnalysisWorkshopSeries_p001_c01
Source: D00000-ElectricVehicleScenarioAnalysisWorkshopSeries.txt | PDF page: 1 | Words: 87

Electric Vehicle Scenario Analysis Workshop Series September - December 2021 Sponsors: Facilitators: Andy Glover, Vice President CoBank’s Beacon Group Tamra Reynolds, Managing Director Luke Gaines, Vice President Dean Church, Vice President CoBank’s Knowledge Exchange Division Teri Viswanath, Lead Economist Presenters: Teri Viswanath, Lead Economist Other Participants: S T R AT E G I C P L A N N I N G A N D Justin Brown Vaughn, Credit Officer FA C I L I TAT I O N S E R V I C E S


## 5. Save one CSV per source file

After inspecting the chunks, run the next cell to save a separate CSV for each source text file. For example, `report.txt` becomes `report_chunks_for_annotation.csv`. Files are saved in the `annotation_csvs` subfolder of your text folder on Google Drive. Each CSV contains only that source's chunks, with blank `statement_status` and `notes` columns. A source with no usable chunks receives a header-only CSV.

Open your assigned CSV in Google Sheets or Excel, turn on text wrapping, and freeze the header row. Use the annotation guide below. Edit only `statement_status` and `notes`, and save your annotated copy under a new name, such as `report_annotations_yourname.csv`. The UTF-8 BOM helps Excel preserve punctuation.

Re-running the save cell replaces these template CSVs. It does not read or preserve labels entered in your spreadsheet, so keep annotated copies under different names.


In [12]:
output_dir = text_folder / "annotation_csvs"  # Save annotation templates in a subfolder on Google Drive.
output_dir.mkdir(parents=True, exist_ok=True)  # Create the output folder if needed.
csv_paths = []  # Store the paths of the saved CSV files.
export_summary = []  # Record the source, row count, and output path for each file.

for filename in sorted(texts):  # Create one CSV for every source text file, in filename order.
    source_rows = dataset.loc[dataset["source_file"] == filename].copy()  # Select only this source's chunks, retaining their order and IDs.
    csv_path = output_dir / f"{Path(filename).stem}_chunks_for_annotation.csv"  # Name the CSV after its source document.
    source_rows.to_csv(csv_path, index=False, encoding="utf-8-sig")  # Save all columns without an extra row-number column.
    csv_paths.append(csv_path)  # Keep the saved path for later use.
    export_summary.append({"source_file": filename, "chunks": len(source_rows), "csv_path": str(csv_path)})  # Record the export details.

print(f"Saved {len(csv_paths)} CSV file(s) to {output_dir}")  # Display the number of files and their destination.
display(pd.DataFrame(export_summary))  # Show which CSV belongs to each source and how many chunks it contains.


Saved 4 CSV file(s) to /content/drive/MyDrive/a-ncsu-courses/Temporary/annotation_csvs


,source_file,chunks,csv_path
0,D00000-ElectricVehicleScenarioAnalysisWorkshop...,38,/content/drive/MyDrive/a-ncsu-courses/Temporar...
1,D00001-TheEraOfFlatPowerDemandIsOver.txt,52,/content/drive/MyDrive/a-ncsu-courses/Temporar...
2,D00002-CharacteristicsAndRiskOfEmergingLargeLo...,135,/content/drive/MyDrive/a-ncsu-courses/Temporar...
3,D00003-PlanningForAndManagingInterminateElecri...,46,/content/drive/MyDrive/a-ncsu-courses/Temporar...


## 6. Use one annotation dimension: Statement status

First look for a **concrete action intended to manage, accommodate, or respond to electricity demand or load growth**. A forecast, risk, broad goal, or technology name alone is not an action. Code what the passage says, without checking whether the action happened in the real world.

Enter one of these exact codes in `statement_status`:

| Code | Meaning from the guide | Invented practice example |
|---|---|---|
| `recommendation` | The passage advises or recommends an action. | Utilities should offer managed charging programs. |
| `plan` | An actor intends or commits to a future action. | The utility plans to launch a managed charging program next year. |
| `practice` | An action is underway, operating, or completed. | The utility launched its managed charging program last year. |
| `general` | The action is described as general, hypothetical, conditional, or of unclear status. | A managed charging program could reduce evening demand. |

Use two additional **classroom handling codes** when a row cannot receive one of those four labels:

- `no_action`: no concrete load-management action appears in the chunk. Examples include a cover, references, or a load forecast alone.
- `review`: the text is damaged, needed context is missing, or multiple actions/statuses prevent a single clear label. Briefly explain in `notes`. Do not guess from words such as “will” or “should” alone.

Several actions with the same clear status can receive that status. When statuses differ, use `review`. For `general`, a recognizable action must still be present; use `review` for unreadable or insufficient evidence.

**Classroom adaptation:** The source is CARESL's `annotation_specs/v0.1/annotation-manual.md`, section “Statement status.” The research guide allows multiple statuses on a strategy instance. Here we label whole chunks with one code and use `review` for mixed cases. The two handling codes are teaching additions, not original status categories. Chunk counts are not counts of distinct strategies.

Open the saved CSV for your assigned source file. Start with **10 instructor-assigned chunks**. Two students can independently annotate the same chunk IDs and discuss differences. Leave unreviewed rows blank. Keep the `text`, IDs, and source fields unchanged; write comments in `notes`. If checking the original resolves an ambiguity, record the context you used in `notes`.